> 极致的工程优化。

- resources
    - https://kellerjordan.github.io/posts/muon/
    - https://github.com/KellerJordan/cifar10-airbench

- a type of second-order optimizer
- New "Muon" optimizer: Trains AI language models faster, using less data.
    - Pareto frontier
- "Muon" excels at training AI efficiently with **very large data batches**.
- "Telescoping"(伸缩) algorithm: Smartly finds optimal AI settings much faster.
    - weight decay, learning rate

- nats
    - $\ln$：以 e 为底（香浓熵以 2 为底）
    - 1.3：$Loss = -\ln(p_{correct})$
        - $p_\text{correct} = e^{-1.3}$ = 27.25%

## basics

- sgd：沿着负梯度方向更新；
- AdamW 则为每个参数计算自适应的学习率
    - 更新一阶矩（动量 Momentum）：$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$
    - 更新二阶矩（方差 Variance）：$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$
    - 偏差校正 (Bias Correction): $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
    - 核心更新：$\Delta W_t' = \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$
        - $RMS(\Delta W_t') = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\Delta W_{t,i}')^2}$
    - 最终更新 $W_t = W_{t-1} - (\Delta W_t' + \lambda W_{t-1})$
- muon
    - $W_{t+1} = W_t - \eta_t O_t$
        - $O_t$ 是对梯度动量矩阵 $M_t$ 进行正交化后得到的更新矩阵。
    - $\eta_t$：
        - moonshoot 
            - "1. 0 to 33B tokens: In this stage, the learning rate linearly increases to 4.2e-4 in 2k steps." （这正是线性预热阶段）
            - "2. 33B to 5.2T tokens: In this stage, the learning rate decays from 4.2e-4 to 4.2e-5 in a cosine style." （这正是余弦衰减阶段）
        - Essential AI
            - "In all runs, we use the learning rate schedule of a linear warmup and cosine decay to 0.1 of the max learning rate." 

### why、when、how Muon

- And for an empirically-flavored motivation, we observe that based on manual inspection, the updates produced by both SGD-momentum and Adam for the 2D parameters in transformer-based neural networks typically have very **high condition number**. That is, they are almost **low-rank matrices**, with the updates for all neurons being dominated by just a few directions. We speculate that orthogonalization effectively increases the scale of other “rare directions” which have small magnitude in the update but are nevertheless important for learning.
    - 神经网络中2D参数的更新通常具有非常高的条件数。也就是说，它们几乎是低秩矩阵，所有神经元的更新都被少数几个方向主导。”
    - 正交化会“有效地增加其他‘稀有方向’的尺度”。这意味着，尽管次要任务的原始梯度信号很弱，但在正交化之后，这个方向在最终的 update 矩阵中会被放大，其重要性得到提升。
- When training a neural network with Muon, scalar and vector parameters of the network (`bias`, `head`), as well as the input and output layers, should be optimized by a standard method such as AdamW. Muon can be used for 4D convolutional parameters by flattening their last three dimensions.

## scaling Muon

- 权重和激活值持续增长
    -  在大规模训练中，原始 Muon 会导致模型权重和层输出的均方根（RMS）持续增大，超出 bf16 浮点数的表示范围，从而损害模型性能。 解决方案：引入权重衰减（Weight Decay） 与 AdamW 类似，为 Muon 引入权重衰减机制，其更新规则变为：
    -  $W_{t+1} = W_t - \eta_t (O_t + \lambda W_t)$
-  不同形状矩阵的更新步长不一致
    -  Muon 的理论更新步长（RMS）与权重矩阵的形状相关，具体为 $\sqrt{1 / \max(A, B)}$（其中 A, B 为矩阵维度）。这意味着，对于不同形状的层（如 MLP 层和注意力头），更新的幅度会不一致，导致训练不稳定和次优性能。 解决方案：实现一致的更新均方根（Consistent Update RMS） 这是 Muon 能够即插即用的关键一步。通过在更新时乘以一个缩放因子 $\sqrt{\max(A, B)}$，可以抵消形状带来的影响。同时，为了匹配 AdamW 的经验更新幅度，再乘以一个常数（如 0.2）。最终的更新规则调整为：
    -  $W_{t+1} = W_t - \eta_t (0.2 \cdot O_t \cdot \sqrt{\max(A, B)} + \lambda W_t)$
-  与 ZeRO-1 分布式策略不兼容
    -  ZeRO-1 等分布式策略会将优化器状态和梯度分片到不同设备上，但 Muon 的正交化需要完整的梯度矩阵，这产生了冲突。 解决方案：开发高效的分布式 Muon 通过在标准的 ZeRO-1 流程中增加一个“DP Gather”操作（在数据并行组内收集完整的梯度分片），可以在计算更新前临时重构出完整的梯度矩阵。虽然这会带来一些额外的通信开销，但实验证明这个开销很小（约 1%-3% 的端到端延迟），在工程上完全可以接受。

### shape matters

- 对于一个形状为 `[A, B]` 的满秩矩阵，经过理想正交化后，其更新矩阵的均方根（Root Mean Square, RMS）为 $\sqrt{1/\max(A,B)}$
    - MLP: MLP 层的权重矩阵非常“胖”。例如，一个隐藏层维度为 4096，MLP 中间层维度为 11008 的矩阵，其形状为 `[4096, 11008]`。
        - $1 / \sqrt{11008} \approx 1 / 105 \approx 0.0095$，这是一个非常小的更新幅度。
        - 对于 MLP 层（$\max(A, B)$ 很大）：更新步长天生就过小。这会导致这些层学习得非常缓慢，模型可能无法充分发挥其表示能力，最终导致性能次优（suboptimal performances）。
    - Attention: 单个注意力头的 KV 权重矩阵可能要“瘦”得多。例如，其形状可能是 `[4096, 128]`（假设 head dimension 为 128）。
        - $1 / \sqrt{4096} = 1 / 64 \approx 0.0156$
        - 对于注意力头（$\max(A, B)$ 很小）：更新步长天生就过大。这会导致训练不稳定，参数更新可能会“矫枉过正”，甚至导致梯度爆炸和训练崩溃。
$$
W_{t+1} = W_t - \eta_t (0.2 \cdot O_t \cdot \sqrt{\max(A, B)} + \lambda W_t)
$$

- $RMS(O_t \cdot \sqrt{\max(A, B)}) \propto \frac{1}{\sqrt{\max(A, B)}} \cdot \sqrt{\max(A, B)} = 1$
- Moonshot 论文（第 4 页） 和 Essential AI 论文（第 3 页） 都提到了这一点：通过大量实验观察，AdamW 在实际训练中的更新 RMS 通常在 0.2 到 0.4 这个范围内。

In [1]:
import numpy as np
A, B = 3, 5
shape = (A, B)
M = np.random.randn(*shape)
rank = np.linalg.matrix_rank(M)
rank

np.int64(3)

In [3]:
U, s, Vh = np.linalg.svd(M, full_matrices=False)
O = U @ Vh
O

array([[ 0.24697648,  0.27606594, -0.82708081,  0.38698815, -0.17019905],
       [-0.78410449,  0.37569842, -0.10332775,  0.20412541,  0.43782077],
       [-0.46044038, -0.59980492, -0.46955856, -0.3874162 , -0.24010863]])

In [5]:
actual_rms = np.sqrt(np.mean(np.square(O)))
actual_rms

np.float64(0.44721359549995804)

In [6]:
theoretical_rms = np.sqrt(1 / max(A, B))
theoretical_rms

np.float64(0.4472135954999579)

## experiments & results

- 拓展计算-时间权衡的“帕累托前沿”
    - 传统的比较只关注在固定资源下谁更快。但实际场景中，我们常常需要在“设备数量”和“训练时间”之间做权衡。Essential AI 的研究表明，Muon 拓展了 AdamW 的帕累托前沿。这意味着：
        - 无论你的目标是“用更多机器在更短时间内完成训练”，还是“用更少机器花更长时间来节省成本”，Muon 都提供了比 AdamW 更优的选项。
- Muon 之所以能拓展帕累托前沿，根本原因在于它在大批量（Large Batch Size）训练时保持了更高的数据效率。
    - 临界批量大小（Critical Batch Size）：对于任何优化器，当批量大小超过一个临界点后，再增大批量大小带来的训练加速效果会减弱。
    - Token Ratio：研究发现，Muon 的临界批量大小远大于 AdamW。即使在超过临界点后，Muon 达到相同损失所需的训练数据量（tokens）也持续少于 AdamW（约少 10-15%），并且这个优势随着批量增大而保持甚至扩大。这意味着使用 Muon，你可以更放心地通过增加并行设备和批量大小来暴力缩短训练时间，而不用太担心数据效率的损失。

两篇论文都展示了 Muon 令人信服的实验结果：

- 计算效率：Moonshot AI 的扩展定律实验表明，在达到相同模型性能（LM Loss）的前提下，Muon 所需的训练计算量（FLOPs）大约只有 AdamW 的 52%。这是一个巨大的效率提升。
- 模型性能：基于 Muon 训练的 Moonlight 模型（一个 3B/16B 的 MoE 模型），在 MMLU 等多个权威基准上，性能显著优于使用 AdamW 训练的同等规模甚至更大规模的模型，刷新了性能的帕累托前沿。
- 更新多样性：通过分析权重矩阵的 SVD 熵，发现 Muon 优化后的矩阵奇异值分布更平坦。这证实了 Muon 确实在更广泛的方向上进行更新，而不是集中于少数几个方向。这对需要激活不同专家（Expert）的 MoE 模型尤其有利。